# DRACO Lite: a small retrieval-aware comparison

This notebook exercises both retrieval routes through the public ScreamingFace SDK. The Engine
owns the Case, retrieval policy, Judge, Grading, and Aggregation; each SDK Candidate owns only its
answer policy.

`draco/lite` uses two pinned representative Cases, ten criteria per Case, and one Judge pass per
criterion. It is useful for directional development checks, but its score is **not comparable**
to canonical DRACO.

> **Spend warning:** execution is disabled by default. Review the discovered Benchmark and set
> `RUN_EVALUATION = True` deliberately; **Run All** otherwise makes no model calls.

## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

Export `TAVILY_API_KEY` before `just stack-up`. A missing key fails the Tavily
Candidate before its first paid model request instead of silently running without retrieval. The
connection panel sends the OpenRouter key through the Engine to AI Gateway; the Client never calls
AI Gateway directly.

In [ ]:
import screamingface as sf

## Connect OpenRouter

In [ ]:
sf.connect()

## Define one Candidate per retrieval route

In [ ]:
DRACO_ANSWER_PROMPT = (
    "You are answering a research-quality prompt. Provide a thorough, "
    "well-reasoned answer in prose. Address every aspect the prompt raises. "
    "Use clear structure (headings, bullet lists where appropriate) and cite "
    "specific facts, methodologies, or sources where relevant.\n\n"
    "Do not refuse, abstain, or claim uncertainty unless the question is "
    "genuinely ambiguous — the goal is to demonstrate depth of understanding. "
    "Length: aim for the level of detail the question warrants; brevity that "
    "skips key points will be penalised by the rubric."
)

In [ ]:
DRACO_PARAMS = {"max_tokens": 8192, "temperature": 0.0}
DRACO_PARAMS_NO_TEMPERATURE = {"max_tokens": 8192}

native_search = sf.Model(
    "openrouter/openai/gpt-5.5",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS_NO_TEMPERATURE,
)
tavily_search = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS,
)

## Evaluate DRACO Lite

The same Engine-owned lite protocol invokes both Candidates. The current reference deployment
routes GPT through provider-native search and Gemini Flash through the guarded Tavily tool loop.
Success proves both configured routes were available; a model may legitimately answer without
calling an offered function. The repository's forced-tool tests certify actual Tavily
`/search` and `/extract` dispatch deterministically.

In [ ]:
RUN_EVALUATION = False

candidates = [native_search, tavily_search]
report = sf.evaluate(candidates, benchmark="draco/lite") if RUN_EVALUATION else None
report_output = (
    report.to_json()
    if report is not None
    else "Evaluation disabled — set RUN_EVALUATION = True to spend."
)
report_output

## Inspect the Report

In [ ]:
if report is not None:
    comparison = [
        {
            "name": result.name,
            "score": result.score,
            "metrics": dict(result.metrics),
            "duration_ms": result.duration_ms,
            "finish_reasons": [case.finish_reason for case in result.cases],
            "failures": result.failures,
            "usage": result.usage,
        }
        for result in report.candidates
    ]
else:
    comparison = []
comparison

In [ ]:
if report is not None:
    selected = report.candidates[0]
    selected_case = selected.cases[0]
    detail = {
        "compiled_url4": selected.url4,
        "operations": selected.operations,
        "case_id": selected_case.case_id,
        "input": selected_case.input,
        "output": selected_case.output,
        "finish_reason": selected_case.finish_reason,
        "grade": selected_case.grade,
        "checks": () if selected_case.grade is None else selected_case.grade.checks,
        "failures": selected_case.failures,
    }
else:
    detail = None
detail

In [ ]:
{
    "ok": report.ok,
    "benchmark": report.benchmark,
    "case_count": report.case_count,
    "duration_ms": report.duration_ms,
    "failures": report.failures,
    "usage": report.usage,
} if report is not None else None

## Corrective loop on DRACO

The same `sf.CorrectiveLoop` from notebook 07 — **one changed `benchmark=` line**. That is the
whole benchmark-independence claim: the loop protocol is a Candidate strategy, so it works on any
Benchmark that advertises a check surface, with no protocol-specific code on either side.

What DRACO's check surface means here, in one line: a draft **passes** when its normalized
weighted rubric score reaches `draco-pass.v1`'s threshold (0.7), and the feedback a failing draft
gets back names only the rubric **areas** that fell short — never a criterion, because the rubric
is the answer key.

> **Spend warning:** unlike IFEval, every DRACO check is a paid Judge call. A two-member,
> three-round loop invokes the check surface up to 6 times per Case *on top of* the members' own
> answers. Each check can retry according to DRACO's bounded policy, so the SDK warns with both
> the check ceiling and that retry caveat before the first request. Leave `RUN_CORRECTIVE = False`
> unless you mean to spend.

In [ ]:
RUN_CORRECTIVE = False

corrective = sf.CorrectiveLoop(
    [native_search, tavily_search],
    judge=sf.Model("openrouter/openai/gpt-5.5", params=DRACO_PARAMS_NO_TEMPERATURE),
    max_rounds=2,
)
corrective_report = (
    sf.evaluate(corrective, benchmark="draco/lite", limit=1) if RUN_CORRECTIVE else None
)
corrective_report if corrective_report is not None else (
    "Corrective loop disabled — set RUN_CORRECTIVE = True to spend on paid checks."
)